In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Given two matrices <code>a</code> and <code>x</code>, each of shape <code>[B, L]</code> (batch size &times; sequence length),
  compute the linear recurrence <code>h</code> of shape <code>[B, L]</code> defined by:
  <code>h[b, 0] = x[b, 0]</code> and <code>h[b, t] = a[b, t] &times; h[b, t&minus;1] + x[b, t]</code> for <code>t &ge; 1</code>.
  All values are <code>float32</code>. This operation is the core computational primitive of
  State Space Models (SSMs) such as Mamba, S4, and H3.
</p>

<svg width="640" height="200" viewBox="0 0 640 200" xmlns="http://www.w3.org/2000/svg"
     style="display:block; margin:20px auto;">
  <rect width="640" height="200" fill="#222" rx="8"/>
  <!-- Title -->
  <text x="320" y="24" fill="#ccc" font-size="13" text-anchor="middle" font-family="monospace">Linear Recurrence: h[t] = a[t] · h[t-1] + x[t]</text>
  <!-- Boxes for h values -->
  <rect x="40"  y="80" width="80" height="40" rx="4" fill="#1a3a5c" stroke="#4a9eff" stroke-width="1.5"/>
  <rect x="180" y="80" width="80" height="40" rx="4" fill="#1a3a5c" stroke="#4a9eff" stroke-width="1.5"/>
  <rect x="320" y="80" width="80" height="40" rx="4" fill="#1a3a5c" stroke="#4a9eff" stroke-width="1.5"/>
  <rect x="460" y="80" width="80" height="40" rx="4" fill="#1a3a5c" stroke="#4a9eff" stroke-width="1.5"/>
  <text x="80"  y="105" fill="#4a9eff" font-size="13" text-anchor="middle" font-family="monospace">h[0]</text>
  <text x="220" y="105" fill="#4a9eff" font-size="13" text-anchor="middle" font-family="monospace">h[1]</text>
  <text x="360" y="105" fill="#4a9eff" font-size="13" text-anchor="middle" font-family="monospace">h[2]</text>
  <text x="500" y="105" fill="#4a9eff" font-size="13" text-anchor="middle" font-family="monospace">h[3]</text>
  <!-- Arrows between h values with a[t] labels -->
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
      <path d="M0,0 L0,6 L8,3 Z" fill="#7ec8a0"/>
    </marker>
  </defs>
  <line x1="120" y1="100" x2="176" y2="100" stroke="#7ec8a0" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="148" y="94" fill="#7ec8a0" font-size="11" text-anchor="middle" font-family="monospace">×a[1]</text>
  <line x1="260" y1="100" x2="316" y2="100" stroke="#7ec8a0" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="288" y="94" fill="#7ec8a0" font-size="11" text-anchor="middle" font-family="monospace">×a[2]</text>
  <line x1="400" y1="100" x2="456" y2="100" stroke="#7ec8a0" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="428" y="94" fill="#7ec8a0" font-size="11" text-anchor="middle" font-family="monospace">×a[3]</text>
  <!-- x[t] inputs from below -->
  <line x1="80"  y1="152" x2="80"  y2="124" stroke="#ccc" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="220" y1="152" x2="220" y2="124" stroke="#ccc" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="360" y1="152" x2="360" y2="124" stroke="#ccc" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="500" y1="152" x2="500" y2="124" stroke="#ccc" stroke-width="1.2" marker-end="url(#arr)"/>
  <text x="80"  y="170" fill="#ccc" font-size="12" text-anchor="middle" font-family="monospace">x[0]</text>
  <text x="220" y="170" fill="#ccc" font-size="12" text-anchor="middle" font-family="monospace">x[1]</text>
  <text x="360" y="170" fill="#ccc" font-size="12" text-anchor="middle" font-family="monospace">x[2]</text>
  <text x="500" y="170" fill="#ccc" font-size="12" text-anchor="middle" font-family="monospace">x[3]</text>
  <!-- Plus signs -->
  <text x="80"  y="147" fill="#ccc" font-size="13" text-anchor="middle" font-family="monospace">+</text>
  <text x="220" y="147" fill="#ccc" font-size="13" text-anchor="middle" font-family="monospace">+</text>
  <text x="360" y="147" fill="#ccc" font-size="13" text-anchor="middle" font-family="monospace">+</text>
  <text x="500" y="147" fill="#ccc" font-size="13" text-anchor="middle" font-family="monospace">+</text>
  <!-- Ellipsis -->
  <text x="590" y="105" fill="#ccc" font-size="18" text-anchor="middle" font-family="monospace">…</text>
</svg>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The result must be stored in the output tensor <code>h</code></li>
</ul>

<h2>Examples</h2>

<p>Example 1 — exponential decay (<code>a = 0.5</code>, single impulse):</p>
$$
a = \begin{bmatrix} 0.5 & 0.5 & 0.5 & 0.5 \end{bmatrix}, \quad
x = \begin{bmatrix} 1.0 & 0.0 & 0.0 & 0.0 \end{bmatrix}
$$
$$
h = \begin{bmatrix} 1.0 & 0.5 & 0.25 & 0.125 \end{bmatrix}
$$

<p>Example 2 — prefix sum (<code>a = 1</code>, unit inputs):</p>
$$
a = \begin{bmatrix} 1.0 & 1.0 & 1.0 & 1.0 \end{bmatrix}, \quad
x = \begin{bmatrix} 1.0 & 1.0 & 1.0 & 1.0 \end{bmatrix}
$$
$$
h = \begin{bmatrix} 1.0 & 2.0 & 3.0 & 4.0 \end{bmatrix}
$$

<p>Full example with <code>B = 2</code>, <code>L = 4</code>:</p>
$$
a = \begin{bmatrix} 0.5 & 0.5 & 0.5 & 0.5 \\ 1.0 & 1.0 & 1.0 & 1.0 \end{bmatrix}, \quad
x = \begin{bmatrix} 1.0 & 0.0 & 0.0 & 0.0 \\ 1.0 & 1.0 & 1.0 & 1.0 \end{bmatrix}
$$
$$
h = \begin{bmatrix} 1.0 & 0.5 & 0.25 & 0.125 \\ 1.0 & 2.0 & 3.0 & 4.0 \end{bmatrix}
$$

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>B</code> &le; 256 (batch size)</li>
  <li>1 &le; <code>L</code> &le; 65,536 (sequence length)</li>
  <li>All values in <code>a</code> and <code>x</code> are <code>float32</code></li>
  <li>Performance is measured with <code>B</code> = 64, <code>L</code> = 16,384</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// a, x, h are device pointers
extern "C" void solve(const float* a, const float* x, float* h, int B, int L) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# a, x, h are tensors on the GPU
@cute.jit
def solve(a: cute.Tensor, x: cute.Tensor, h: cute.Tensor, B: cute.Int32, L: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# a, x are tensors on GPU
@jax.jit
def solve(a: jax.Array, x: jax.Array, B: int, L: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# a, x, h are device pointers
@export
def solve(
    a: UnsafePointer[Float32, MutExternalOrigin],
    x: UnsafePointer[Float32, MutExternalOrigin],
    h: UnsafePointer[Float32, MutExternalOrigin],
    B: Int32,
    L: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# a, x, h are tensors on the GPU
def solve(a: torch.Tensor, x: torch.Tensor, h: torch.Tensor, B: int, L: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# a, x, h are tensors on the GPU
def solve(a: torch.Tensor, x: torch.Tensor, h: torch.Tensor, B: int, L: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Linear Recurrence",
            atol=1e-05,
            rtol=1e-05,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        a: torch.Tensor,
        x: torch.Tensor,
        h: torch.Tensor,
        B: int,
        L: int,
    ):
        assert a.shape == (B, L)
        assert x.shape == (B, L)
        assert h.shape == (B, L)
        assert a.dtype == x.dtype == h.dtype == torch.float32
        assert a.device.type == "cuda"
        assert x.device.type == "cuda"
        assert h.device.type == "cuda"

        out = torch.empty_like(x)
        out[:, 0] = x[:, 0]
        for t in range(1, L):
            out[:, t] = a[:, t] * out[:, t - 1] + x[:, t]
        h.copy_(out)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "a": (ctypes.POINTER(ctypes.c_float), "in"),
            "x": (ctypes.POINTER(ctypes.c_float), "in"),
            "h": (ctypes.POINTER(ctypes.c_float), "out"),
            "B": (ctypes.c_int, "in"),
            "L": (ctypes.c_int, "in"),
        }

    def _make_test_case(self, B, L, zero_inputs=False, zero_a=False, unit_a=False):
        device = "cuda"
        dtype = torch.float32
        if zero_inputs:
            a = torch.zeros(B, L, device=device, dtype=dtype)
            x = torch.zeros(B, L, device=device, dtype=dtype)
        elif zero_a:
            a = torch.zeros(B, L, device=device, dtype=dtype)
            x = torch.randn(B, L, device=device, dtype=dtype)
        elif unit_a:
            a = torch.ones(B, L, device=device, dtype=dtype)
            x = torch.randn(B, L, device=device, dtype=dtype)
        else:
            a = torch.rand(B, L, device=device, dtype=dtype)
            x = torch.randn(B, L, device=device, dtype=dtype)
        h = torch.empty(B, L, device=device, dtype=dtype)
        return {"a": a, "x": x, "h": h, "B": B, "L": L}

    def generate_example_test(self) -> Dict[str, Any]:
        device = "cuda"
        dtype = torch.float32
        a = torch.tensor(
            [[0.5, 0.5, 0.5, 0.5], [1.0, 1.0, 1.0, 1.0]],
            device=device,
            dtype=dtype,
        )
        x = torch.tensor(
            [[1.0, 0.0, 0.0, 0.0], [1.0, 1.0, 1.0, 1.0]],
            device=device,
            dtype=dtype,
        )
        h = torch.empty(2, 4, device=device, dtype=dtype)
        return {"a": a, "x": x, "h": h, "B": 2, "L": 4}

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        torch.manual_seed(42)
        tests = []

        # Edge case: single element
        tests.append(self._make_test_case(1, 1))

        # Edge case: two elements
        tests.append(self._make_test_case(1, 2))

        # Zero inputs
        tests.append(self._make_test_case(4, 4, zero_inputs=True))

        # a=0 everywhere: h[t] = x[t] (no recurrence)
        tests.append(self._make_test_case(4, 16, zero_a=True))

        # a=1 everywhere: h[t] = prefix sum of x
        tests.append(self._make_test_case(4, 16, unit_a=True))

        # Power-of-2 sequence length
        tests.append(self._make_test_case(8, 32))

        # Power-of-2 sequence length, larger
        tests.append(self._make_test_case(8, 256))

        # Non-power-of-2 sequence length
        tests.append(self._make_test_case(4, 30))

        # Non-power-of-2 sequence length, larger
        tests.append(self._make_test_case(8, 100))

        # Realistic size (SSM hidden state)
        tests.append(self._make_test_case(16, 1024))

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        torch.manual_seed(0)
        # B=64 sequences, L=16384 tokens — typical long-context SSM workload
        return self._make_test_case(64, 16384)


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
